In [1]:
# --- Parameters ---

# Env - MUST MATCH your training notebooks
NUM_AGENTS = 2
HEIGHT = 6
WIDTH = 6
SPAWN_PROB_PER_CELL = 0.05
DESPAWN_PROB_PER_CELL = 0.09
DISCOUNT = 0.99

# Monte Carlo Simulation
NUM_MONTE_CARLO_SAMPLES = 40000  # Number of episodes to average over for each state
EPISODE_LENGTH = 1000           # How many steps to simulate into the future per episode


In [2]:

# --- Imports ---
import sys
sys.path.append("../..") # Adjust this path if your notebook is in a different location
import numpy as np
from tqdm import tqdm
import pickle

# Assumes your helper files are in a 'tadd_helpers' subfolder
from tadd_helpers.env_functions import State, spawn_apples, despawn_apples
from tadd_helpers.eval_functions import symmetric_nearest_policy

In [3]:
def update_move_state(s: State, agent_idx: int, move_vector: np.ndarray):
    """Simulates the 'move' half-step. Returns: (next_state, rewards_list)"""
    old_agent_pos = s.agent_position(agent_idx)
    new_agent_pos = np.clip(old_agent_pos + move_vector, [0, 0], [s.H - 1, s.L - 1])
    s_new = s.copy()
    s_new.set_agent_position(agent_idx, new_agent_pos)
    rewards_list: list[float] = [0.0 for _ in range(NUM_AGENTS)]
    return s_new, rewards_list

def update_spawn_despawn_and_pick(s: State, agent_idx: int, picked: bool):
    """Simulates the 'pick/spawn' half-step. Returns: (next_state, rewards_list)"""
    rewards: list[float] = [0.0 for _ in range(NUM_AGENTS)]
    s_new = s.copy()
    if picked:
        picker_pos = s_new.agent_position(agent_idx)
        s_new.remove_apple_at(picker_pos)
        rewards[agent_idx] = -1.0
        # For the 2-agent case, the other agent's reward must be +2
        for other_idx in range(NUM_AGENTS):
            if other_idx != agent_idx:
                rewards[other_idx] = 2.0

    spawn_apples(s_new, SPAWN_PROB_PER_CELL)
    despawn_apples(s_new, DESPAWN_PROB_PER_CELL)
    return s_new, rewards

In [4]:
# --- Define the Specific State for Evaluation ---
states_to_evaluate: list[State] = []

# The state defined in your hypothesis
# Picker (A0) at (0,0), Observer (A1) at (5,5) - geometrically even
symmetric_apples = np.zeros((HEIGHT, WIDTH), dtype=int)
symmetric_apples[0, 0] = 1
symmetric_agents = {0: (0, 0), 1: (5, 5)} # Agent 0 is the picker

symmetric_test_state = State(_apples=symmetric_apples, _agents=symmetric_agents, name="Symmetric Test State")
states_to_evaluate.append(symmetric_test_state)

# --- Print the state to verify it is correct ---
print("State to be evaluated:")
print(symmetric_test_state)

State to be evaluated:
--- Symmetric Test State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (0, 0)
  Agent 1: (5, 5)

--- Agents (Count) ---
1 . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . 1

--- Apples (Count) ---
1 . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .


In [5]:
# This dictionary will store the results
value_results = {}

for start_state in states_to_evaluate:
    print(f"\n--- Running Monte Carlo for state: '{start_state.name}' ---")
    
    agent_value_samples = []

    # Get the ID of the picker agent from the start state definition
    # We assume the agent at (0,0) is the picker, which is Agent 0
    PICKER_AGENT_ID = 0

    for i in tqdm(range(NUM_MONTE_CARLO_SAMPLES), desc="Simulating Episodes"):
        
        current_half_state = start_state.copy()
        total_discounted_rewards = np.zeros(NUM_AGENTS)
        
        # --- First Action: The t+0.5 -> t+1 Transition (NO DISCOUNT) ---
        
        # --- THIS IS THE FIX ---
        # The actor is KNOWN to be the picker. There is no random choice.
        agent_to_act_first_step = PICKER_AGENT_ID
        # --------------------
        
        picked = current_half_state.apples[current_half_state.agent_position_tuple(agent_to_act_first_step)] >= 1
        assert picked, "The designated picker agent is not on an apple in the start state!"

        s_next, r_pick_list = update_spawn_despawn_and_pick(current_half_state, agent_to_act_first_step, picked)
        
        # Accumulate the immediate rewards (at t=0, discount is 1.0)
        total_discounted_rewards += np.array(r_pick_list)
        
        # --- Main Episode Loop: Simulate from t+1 onwards ---
        current_state = s_next
        step_discount = DISCOUNT # Discount for the first full step (t=1)

        for step in range(EPISODE_LENGTH):
            # In the subsequent loop, the agent is chosen randomly as normal
            agent_to_act_loop = current_state.get_random_agent_id()
            move_action = symmetric_nearest_policy(current_state, agent_to_act_loop)
            
            # First half-step (Move)
            s_half_loop, r_move_list_loop = update_move_state(current_state, agent_to_act_loop, move_action.vector)
            
            # Second half-step (Pick/Spawn)
            picked_loop = s_half_loop.apples[s_half_loop.agent_position_tuple(agent_to_act_loop)] >= 1
            s_next_loop, r_pick_list_loop = update_spawn_despawn_and_pick(s_half_loop, agent_to_act_loop, picked_loop)
            
            # Correctly calculate and apply discount for the full step
            total_step_reward = np.array(r_move_list_loop) + np.array(r_pick_list_loop)
            total_discounted_rewards += step_discount * total_step_reward
            step_discount *= DISCOUNT

            current_state = s_next_loop

        agent_value_samples.append(total_discounted_rewards)

    # Calculate the final mean and std dev
    mean_values = np.mean(agent_value_samples, axis=0)
    std_dev_values = np.std(agent_value_samples, axis=0)
    value_results[start_state.name] = (mean_values, std_dev_values)


--- Running Monte Carlo for state: 'Symmetric Test State' ---


Simulating Episodes:   0%|          | 0/40000 [00:00<?, ?it/s]

Simulating Episodes:   0%|          | 167/40000 [00:08<35:15, 18.83it/s]


KeyboardInterrupt: 

In [ ]:
print("\n--- Final Analysis ---")

# --- Retrieve Results ---
results = value_results.get("Symmetric Test State")
mean_vals, std_vals = results
picker_val = mean_vals[0]
observer_val = mean_vals[1]
experimental_diff = observer_val - picker_val

# --- Theoretical Value (under PASSIVE observer assumption) ---
n = NUM_AGENTS
gamma = DISCOUNT
theoretical_passive_diff = (2 - (-1)) / (n - (n - 1) * gamma)

# --- Print Comparison ---
print("\n--- Verifying Hypothesis for the Symmetric State ---")
print(f"  - Picker Value (A0):     {picker_val:.4f} ± {std_vals[0]:.4f}")
print(f"  - Observer Value (A1):   {observer_val:.4f} ± {std_vals[1]:.4f}")
print("-" * 60)
print(f"  - EXPERIMENTAL Difference (under nearest_policy): {experimental_diff:.4f}")


--- Final Analysis ---

--- Verifying Hypothesis for the Symmetric State ---
  - Picker Value (A0):     26.2759 ± 7.1652
  - Observer Value (A1):   29.6330 ± 7.1018
------------------------------------------------------------
  - EXPERIMENTAL Difference (under nearest_policy): 3.3571
